## Generating  Customers Data 


In [1]:
import pandas as pd
import random
from faker import Faker

fake = Faker("en_NG")
num_records = 1000

data = []

# Valid Nigerian prefixes
valid_prefixes = ["070", "080", "081", "090", "091", "071"]

def generate_valid_phone():
    prefix = random.choice(valid_prefixes)
    remaining_digits = "".join(str(random.randint(0, 9)) for _ in range(8))
    return prefix + remaining_digits  # total = 11 digits

for _ in range(num_records):
    record = {
        "name": fake.name() if random.random() > 0.1 else None,  # 10% missing

        # ✅ Phone numbers: valid or missing only
        "phone_number": (
            generate_valid_phone() if random.random() > 0.1 else None
        ),

        # IDs: mostly valid, some invalid
        "id_number": (
            str(fake.random_number(digits=11, fix_len=True))
            if random.random() > 0.2
            else str(fake.random_number(digits=random.choice([8, 9, 12, 13])))
        ),

        "address": fake.address() if random.random() > 0.1 else None,

        "reg_date": (
            fake.date_between(start_date="-2y", end_date="today")
            if random.random() > 0.1 else None
        )
    }

    data.append(record)

df_raw = pd.DataFrame(data)

# Save raw data
df_raw.to_csv("sim_registration_raw.csv", index=False)

print("Raw fake data created successfully")
print(df_raw.head(10))


Raw fake data created successfully
                name phone_number     id_number  \
0      Sarah Nnamani  09107818569   22095317788   
1      Philip Oyekan  09074136836  529924980651   
2           John Obi  09080751062   55134600893   
3     Stephen Chukwu  07118896894   85829978416   
4      Samuel Abiola         None   58917371420   
5      Gloria Okafor  08027238552   62483474984   
6  Victoria Akinwale  08162400937   51628068310   
7       Grace Okafor  08036070595      93292457   
8         Joshua Obi  08165918421   28728740323   
9  Joshua Adetokunbo  08012600750   84178223085   

                                             address    reg_date  
0        42305 Ruth Valleys\nPatienceville, VA 40710  2024-12-09  
1   90551 Eze Grove Suite 538\nAdeyemibury, CO 08159  2024-05-21  
2                        USCGC Okonkwo\nFPO AA 10023  2024-10-12  
3            7073 Grace River\nDanielville, GA 85343  2025-07-29  
4                   PSC 6017, Box 6114\nAPO AE 16864  2025-09-14  
5

In [28]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          895 non-null    object
 1   phone_number  910 non-null    object
 2   id_number     1000 non-null   object
 3   address       923 non-null    object
 4   reg_date      901 non-null    object
dtypes: object(5)
memory usage: 39.2+ KB


## ETL Pipeline

In [ ]:
"""
SIM Registration ETL Pipeline

This script performs a complete ETL (Extract, Transform, Load) workflow
for SIM registration data. It reads raw CSV data, cleans and validates
key fields such as names, phone numbers, ID numbers, addresses, and
registration dates, then loads the cleaned dataset into both a CSV file
and a SQLite database.

The pipeline is designed for automation and reproducibility, ensuring
consistent data quality for downstream analytics and reporting.
"""

import pandas as pd
import sqlite3
import re


# 1. EXTRACTION
print("\n1 EXTRACTING DATA...")

csv_path = "sim_registration_raw.csv"

df = pd.read_csv(
    csv_path,
    dtype={"phone_number": str, "id_number": str},
    na_values=["", "NaN", "NA", "None", "nan"],
)

print(f"Rows extracted: {len(df):,}")
print(df.head())



# 2. TRANSFORMATION
print("\n2 TRANSFORMING DATA...")


# -------- NAME CLEANING --------
df["name"] = (
    df["name"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)


# -------- PHONE NUMBER CLEANING --------
def clean_phone(x):
    """
    Clean and normalize phone numbers into a standard Nigerian format.

    This function removes non-numeric characters, strips trailing '.0'
    artifacts from numeric imports, converts Nigerian country code
    (234) into a local format, and validates that the final number
    contains exactly 11 digits starting with '0'.

    Args:
        x: Raw phone number value from the dataset.

    Returns:
        A cleaned phone number string if valid, otherwise None.
    """
    if pd.isna(x):
        return None

    x = str(x).strip()

    # Remove trailing .0 if it exists
    if x.endswith(".0"):
        x = x[:-2]

    digits = re.sub(r"\D", "", x)

    # Convert Nigerian country code to local format
    if digits.startswith("234") and len(digits) > 3:
        digits = "0" + digits[3:]

    if len(digits) == 11 and digits.startswith("0"):
        return digits

    return None


df["phone_number"] = df["phone_number"].apply(clean_phone)
df = df.dropna(subset=["phone_number"])


# -------- ID NUMBER CLEANING --------
import re
import pandas as pd
import numpy as np


def clean_id(x):
    """
    Clean and validate identification numbers.

    This function removes all non-numeric characters and ensures that
    only valid 11-digit ID numbers are retained.

    Args:
        x: Raw ID number value from the dataset.

    Returns:
        A cleaned 11-digit ID string if valid, otherwise None.
    """
    if pd.isna(x):
        return None

    digits = re.sub(r"\D", "", str(x))

    # Keep only valid 11-digit IDs
    if len(digits) == 11:
        return digits

    return None


# Apply cleaning
df["id_number"] = df["id_number"].apply(clean_id)

# Drop unknown / invalid IDs
df = df.dropna(subset=["id_number"])


# -------- ADDRESS CLEANING --------
df["address"] = (
    df["address"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)


# -------- REGISTRATION DATE --------
df["reg_date"] = pd.to_datetime(
    df["reg_date"],
    errors="coerce",
)

# Fill empty dates with placeholder (1900-01-01)
df["reg_date"] = df["reg_date"].fillna(pd.Timestamp("1900-01-01"))


print("Transformation completed")


# 3. LOAD (CSV)
print("\n3 LOADING CLEAN DATA TO CSV...")

clean_csv = "sim_registration_cleaned.csv"
df.to_csv(clean_csv, index=False)

print(f"Clean CSV saved: {clean_csv}")
print(f"Final rows: {len(df):,}")



# 4. LOAD (DATABASE)

print("\n4 LOADING DATA TO DATABASE...")

conn = sqlite3.connect("sim_registration.db")

df.to_sql(
    "sim_registrations",
    conn,
    if_exists="replace",
    index=False,
)

conn.close()

print("Data successfully loaded into SQLite database")



1️⃣ EXTRACTING DATA...
Rows extracted: 1,000
             name phone_number     id_number  \
0   Sarah Nnamani  09107818569   22095317788   
1   Philip Oyekan  09074136836  529924980651   
2        John Obi  09080751062   55134600893   
3  Stephen Chukwu  07118896894   85829978416   
4   Samuel Abiola          NaN   58917371420   

                                            address    reg_date  
0       42305 Ruth Valleys\nPatienceville, VA 40710  2024-12-09  
1  90551 Eze Grove Suite 538\nAdeyemibury, CO 08159  2024-05-21  
2                       USCGC Okonkwo\nFPO AA 10023  2024-10-12  
3           7073 Grace River\nDanielville, GA 85343  2025-07-29  
4                  PSC 6017, Box 6114\nAPO AE 16864  2025-09-14  

2 TRANSFORMING DATA...
Transformation completed

3 LOADING CLEAN DATA TO CSV...
Clean CSV saved: sim_registration_cleaned.csv
Final rows: 721

4 LOADING DATA TO DATABASE...
Data successfully loaded into SQLite database


In [4]:
df = pd.read_csv("sim_registration_cleaned.csv")
df

,name,phone_number,id_number,address,reg_date
0,Sarah Nnamani,9107818569,22095317788,"42305 Ruth Valleys\nPatienceville, VA 40710",2024-12-09
1,John Obi,9080751062,55134600893,USCGC Okonkwo\nFPO AA 10023,2024-10-12
2,Stephen Chukwu,7118896894,85829978416,"7073 Grace River\nDanielville, GA 85343",2025-07-29
3,Gloria Okafor,8027238552,62483474984,Unknown,2024-05-18
4,Victoria Akinwale,8162400937,51628068310,"385 Joshua Plains\nWest Sarahberg, TN 53338",1900-01-01
...,...,...,...,...,...
716,Andrew Adeyemi,8118082103,34245184564,"107 Faith Loop Apt. 339\nBlessingtown, LA 61516",2024-08-27
717,Joshua Balogun,9024988105,73359013150,"3308 Paul Underpass Suite 107\nJamesmouth, OH ...",2024-03-31
718,Emmanuel Nnamani,9004940449,47088719308,"249 Uche Street Apt. 920\nEast Mary, NM 60703",1900-01-01
719,Faith Olawale,9169942484,33893774728,USNV Eze\nFPO AP 26280,2025-06-18


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 721 entries, 0 to 720
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          721 non-null    object
 1   phone_number  721 non-null    int64 
 2   id_number     721 non-null    int64 
 3   address       721 non-null    object
 4   reg_date      721 non-null    object
dtypes: int64(2), object(3)
memory usage: 28.3+ KB
